In [ ]:
from hcc_release_paths import project_path as _hcc_path
from pathlib import Path
import pandas as pd
import numpy as np
import json
import gzip
import os
import sys
import platform
import urllib.request

SEED = 42
np.random.seed(SEED)

ROOT = Path(_hcc_path('LIHC_scRNA_Step18_GSE149614'))

RAW_DIR = ROOT / "Data" / "Raw"
RESULTS_DIR = ROOT / "Results"
FIGURES_DIR = ROOT / "Figures"
LOGS_DIR = ROOT / "Logs"
NOTEBOOK_DIR = ROOT / "Notebook"

for p in [
    RAW_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    LOGS_DIR,
    NOTEBOOK_DIR
]:
    p.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Python:", sys.version)
print("Platform:", platform.platform())

In [ ]:
packages = [
    "scanpy",
    "anndata",
    "pandas",
    "numpy",
    "scipy",
    "matplotlib",
]

for pkg in packages:
    try:
        mod = __import__(pkg)
        version = getattr(mod, "__version__", "unknown")
        print(f"{pkg:12s} OK   {version}")
    except Exception as e:
        print(f"{pkg:12s} MISSING   {e}")

In [ ]:
from hcc_release_paths import project_path as _hcc_path
from pathlib import Path
import pandas as pd
import numpy as np
import json
import gzip
import os
import sys
import platform
import urllib.request

SEED = 42
np.random.seed(SEED)

ROOT = Path(_hcc_path('LIHC_scRNA_Step18_GSE149614'))

RAW_DIR = ROOT / "Data" / "Raw"
RESULTS_DIR = ROOT / "Results"
FIGURES_DIR = ROOT / "Figures"
LOGS_DIR = ROOT / "Logs"
NOTEBOOK_DIR = ROOT / "Notebook"

for p in [RAW_DIR, RESULTS_DIR, FIGURES_DIR, LOGS_DIR, NOTEBOOK_DIR]:
    p.mkdir(parents=True, exist_ok=True)

dataset_info = {
    "dataset": "GSE149614",
    "species": "Homo sapiens",
    "disease": "Hepatocellular carcinoma",
    "role": "Primary scRNA-seq discovery cohort",
    "analysis_stage": "Step18_Data_Audit"
}

with open(
    RESULTS_DIR / "GSE149614_dataset_info.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=4)

print(dataset_info)

In [ ]:
FILES = {
    "metadata": {
        "url": (
            "https://ftp.ncbi.nlm.nih.gov/geo/series/"
            "GSE149nnn/GSE149614/suppl/"
            "GSE149614_HCC.metadata.updated.txt.gz"
        ),
        "path": RAW_DIR / "GSE149614_HCC.metadata.updated.txt.gz"
    },

    "counts": {
        "url": (
            "https://ftp.ncbi.nlm.nih.gov/geo/series/"
            "GSE149nnn/GSE149614/suppl/"
            "GSE149614_HCC.scRNAseq.S71915.count.txt.gz"
        ),
        "path": RAW_DIR / "GSE149614_HCC.scRNAseq.S71915.count.txt.gz"
    }
}

FILES

In [ ]:
from pathlib import Path

metadata_path = FILES["metadata"]["path"]
counts_path = FILES["counts"]["path"]

print("Metadata file:")
print(metadata_path)
print("Exists:", metadata_path.exists())
if metadata_path.exists():
    print("Size MB:", round(metadata_path.stat().st_size / 1024**2, 2))

print("\nCount matrix file:")
print(counts_path)
print("Exists:", counts_path.exists())
if counts_path.exists():
    print("Size MB:", round(counts_path.stat().st_size / 1024**2, 2))

assert metadata_path.exists(), "Metadata 文件不存在"
assert counts_path.exists(), "Count matrix 文件不存在"

print("\nINPUT FILES = READY")

In [ ]:
meta = pd.read_csv(
    metadata_path,
    sep="\t",
    compression="gzip",
    low_memory=False
)

print("Metadata shape:", meta.shape)

print("\nColumns:")
for i, col in enumerate(meta.columns, 1):
    print(f"{i:02d}. {col}")

display(meta.head())

In [ ]:
print("=" * 80)
print("METADATA AUDIT")
print("=" * 80)

print("\nRows:", meta.shape[0])
print("Columns:", meta.shape[1])

print("\nMissing values:")
display(
    meta.isna()
        .sum()
        .sort_values(ascending=False)
        .head(30)
)

print("\nLow-cardinality fields:")

for col in meta.columns:
    n_unique = meta[col].nunique(dropna=False)

    if n_unique <= 30:
        print("\n" + "=" * 60)
        print(col)
        print("=" * 60)
        print(meta[col].value_counts(dropna=False))

In [ ]:
import gzip

with gzip.open(counts_path, "rt") as f:
    header = f.readline().rstrip("\n\r").split("\t")
    first_line = f.readline().rstrip("\n\r").split("\t")

print("Header fields:", len(header))

print("\nFirst header entries:")
print(header[:10])

print("\nFirst data row:")
print(first_line[:10])

print("\nFirst field of header:")
print(repr(header[0]))

In [ ]:
count_cells = pd.Index(
    [str(x) for x in header[1:]],
    dtype="object"
)

print("Number of cells from count matrix:", len(count_cells))
print("First 10 cell IDs:", count_cells[:10].tolist())
print("Duplicate cell IDs:", int(count_cells.duplicated().sum()))

In [ ]:
count_cell_set = set(count_cells)

barcode_candidates = []

for col in meta.columns:
    values = meta[col].dropna().astype(str)

    overlap = len(
        count_cell_set.intersection(set(values))
    )

    barcode_candidates.append({
        "Column": col,
        "Overlap": overlap,
        "Count_cells": len(count_cells),
        "Metadata_rows": len(meta),
        "Match_pct": overlap / len(count_cells) * 100
    })

barcode_match = (
    pd.DataFrame(barcode_candidates)
    .sort_values(
        ["Overlap", "Match_pct"],
        ascending=False
    )
    .reset_index(drop=True)
)

display(barcode_match.head(15))

In [ ]:
print("=" * 80)
print("STEP18 CURRENT CHECKPOINT SUMMARY")
print("=" * 80)

# 1. Metadata shape
print("\n[1] Metadata shape")
print(meta.shape)

# 2. Metadata columns
print("\n[2] Metadata columns")
for i, col in enumerate(meta.columns, 1):
    print(f"{i:02d}. {col}")

# 3. Number of cells from count matrix
print("\n[3] Number of cells from count matrix")
print(len(count_cells))

print("\nFirst 10 count-matrix cell IDs:")
print(count_cells[:10].tolist())

# 4. barcode match top 15
print("\n[4] barcode_match.head(15)")
display(barcode_match.head(15))

print("\n" + "=" * 80)
print("END OF SUMMARY")
print("=" * 80)

In [ ]:
# ============================================================
# STEP18 - Identify metadata-only cell
# ============================================================

BARCODE_COL = "Cell"

meta_cells = pd.Index(
    meta[BARCODE_COL].astype(str)
)

count_cells = pd.Index(
    count_cells.astype(str)
)

metadata_only_cells = meta_cells.difference(count_cells)
count_only_cells = count_cells.difference(meta_cells)

print("=" * 80)
print("CELL ID CONSISTENCY CHECK")
print("=" * 80)

print("Metadata rows:", len(meta))
print("Metadata unique Cell IDs:", meta_cells.nunique())

print("Count matrix cells:", len(count_cells))
print("Count matrix unique Cell IDs:", count_cells.nunique())

print("\nMetadata-only cells:")
print(metadata_only_cells.tolist())

print("\nCount-only cells:")
print(count_only_cells.tolist())

print("\nDuplicated Cell IDs in metadata:")
print(int(meta_cells.duplicated().sum()))

print("\nDuplicated Cell IDs in count matrix:")
print(int(count_cells.duplicated().sum()))

In [ ]:
BARCODE_COL = "Cell"
SAMPLE_COL = "sample"
SITE_COL = "site"
PATIENT_COL = "patient"
STAGE_COL = "stage"
VIRUS_COL = "virus"
AUTHOR_CELLTYPE_COL = "celltype"

metadata_cells = set(meta[BARCODE_COL].astype(str))
count_cell_set = set(count_cells.astype(str))

metadata_only = sorted(metadata_cells - count_cell_set)
count_only = sorted(count_cell_set - metadata_cells)

discrepancy_df = pd.DataFrame({
    "Type": (
        ["Metadata_only"] * len(metadata_only)
        + ["Count_only"] * len(count_only)
    ),
    "Cell": metadata_only + count_only
})

discrepancy_df.to_csv(
    RESULTS_DIR / "Cell_ID_Discrepancy.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Metadata-only cells:", metadata_only)
print("Count-only cells:", count_only)

print("\nDecision:")
print(
    "Use the 71,914 cells present in the expression matrix; "
    "do not impute or fabricate the missing cell."
)

In [ ]:
meta2 = meta.copy()

meta2[BARCODE_COL] = (
    meta2[BARCODE_COL]
    .astype(str)
)

meta2 = (
    meta2
    .drop_duplicates(subset=BARCODE_COL)
    .set_index(BARCODE_COL)
)

common_cells = [
    cell for cell in count_cells
    if cell in meta2.index
]

match_rate = len(common_cells) / len(count_cells)

print("Count matrix cells:", len(count_cells))
print("Matched metadata cells:", len(common_cells))
print(f"Match rate: {match_rate * 100:.4f}%")

assert len(common_cells) == len(count_cells)
assert match_rate == 1.0

print("\nCELL-METADATA ALIGNMENT = PASS")

In [ ]:
for col in [
    SAMPLE_COL,
    SITE_COL,
    PATIENT_COL,
    STAGE_COL,
    VIRUS_COL,
    AUTHOR_CELLTYPE_COL
]:
    print("\n" + "=" * 70)
    print(col)
    print("=" * 70)

    print(
        meta2.loc[common_cells, col]
        .value_counts(dropna=False)
    )

In [ ]:
sample_audit = (
    meta2.loc[common_cells]
    .groupby(
        [
            SAMPLE_COL,
            PATIENT_COL,
            SITE_COL
        ],
        dropna=False
    )
    .size()
    .reset_index(name="N_cells")
    .sort_values(
        [
            PATIENT_COL,
            SAMPLE_COL
        ]
    )
)

display(sample_audit)

sample_audit.to_csv(
    RESULTS_DIR /
    "GSE149614_Sample_Cell_Counts_Before_QC.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nNumber of patients:",
      meta2.loc[common_cells, PATIENT_COL].nunique())

print("Number of samples:",
      meta2.loc[common_cells, SAMPLE_COL].nunique())

print("Number of sites:",
      meta2.loc[common_cells, SITE_COL].nunique())

In [ ]:
import gzip

with gzip.open(counts_path, "rt") as f:
    header_line = f.readline().rstrip("\n\r")
    first_data_line = f.readline().rstrip("\n\r")

header = header_line.split("\t")
first_data = first_data_line.split("\t")

print("=" * 80)
print("COUNT FILE STRUCTURE CHECK")
print("=" * 80)

print("Header field number:", len(header))
print("First data-row field number:", len(first_data))

print("\nFirst 10 HEADER fields:")
for i, x in enumerate(header[:10]):
    print(i, repr(x))

print("\nFirst 10 DATA fields:")
for i, x in enumerate(first_data[:10]):
    print(i, repr(x))

print("\nFirst header field:")
print(repr(header[0]))

print("\nFirst gene:")
print(repr(first_data[0]))

print("\nNumber of numeric values in first gene row:")
print(len(first_data) - 1)

In [ ]:
count_cells = pd.Index(
    [str(x) for x in header],
    dtype="object"
)

print("Number of cells:", len(count_cells))
print("First 10 cells:", count_cells[:10].tolist())
print("Duplicated cells:", int(count_cells.duplicated().sum()))

In [ ]:
BARCODE_COL = "Cell"

meta_cells = pd.Index(
    meta[BARCODE_COL].astype(str)
)

metadata_only_cells = meta_cells.difference(count_cells)
count_only_cells = count_cells.difference(meta_cells)

print("=" * 80)
print("CORRECTED CELL ID CONSISTENCY CHECK")
print("=" * 80)

print("Metadata rows:", len(meta))
print("Metadata unique cells:", meta_cells.nunique())

print("Count matrix cells:", len(count_cells))
print("Count matrix unique cells:", count_cells.nunique())

print("\nMetadata-only cells:")
print(metadata_only_cells.tolist())

print("\nCount-only cells:")
print(count_only_cells.tolist())

print("\nMetadata duplicates:")
print(int(meta_cells.duplicated().sum()))

print("\nCount duplicates:")
print(int(count_cells.duplicated().sum()))

In [ ]:
meta2 = meta.copy()

meta2[BARCODE_COL] = meta2[BARCODE_COL].astype(str)

assert meta2[BARCODE_COL].duplicated().sum() == 0

meta2 = meta2.set_index(BARCODE_COL)

common_cells = [
    c for c in count_cells
    if c in meta2.index
]

match_rate = len(common_cells) / len(count_cells)

print("Count matrix cells:", len(count_cells))
print("Matched metadata cells:", len(common_cells))
print(f"Match rate: {match_rate * 100:.4f}%")

assert len(common_cells) == len(count_cells)
assert match_rate == 1.0

print("\nCELL-METADATA ALIGNMENT = PASS")

In [ ]:
pd.DataFrame(
    columns=["Type", "Cell"]
).to_csv(
    RESULTS_DIR / "Cell_ID_Discrepancy.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Cell_ID_Discrepancy.csv corrected.")

In [ ]:
import gzip
import numpy as np
from scipy import sparse
import time

gene_names = []
data_parts = []
index_parts = []
indptr = [0]

n_cells_expected = len(count_cells)

start_time = time.time()

print("=" * 80)
print("STREAMING COUNT MATRIX → SPARSE MATRIX")
print("=" * 80)

print("Expected cells:", n_cells_expected)

with gzip.open(counts_path, "rt") as f:

    # 第一行就是全部 cell barcodes
    f.readline()

    for gene_idx, line in enumerate(f):

        line = line.rstrip("\n\r")

        gene, sep, values_str = line.partition("\t")

        if sep == "":
            raise ValueError(
                f"Malformed row at gene index {gene_idx}"
            )

        values = np.fromstring(
            values_str,
            sep="\t",
            dtype=np.int32
        )

        if values.size != n_cells_expected:
            raise ValueError(
                f"Column mismatch at gene {gene}: "
                f"{values.size} != {n_cells_expected}"
            )

        nz = np.flatnonzero(values)

        if nz.size > 0:
            data_parts.append(
                values[nz].astype(
                    np.int32,
                    copy=False
                )
            )

            index_parts.append(
                nz.astype(
                    np.int32,
                    copy=False
                )
            )

        indptr.append(
            indptr[-1] + nz.size
        )

        gene_names.append(gene)

        if (gene_idx + 1) % 1000 == 0:
            elapsed = (
                time.time() - start_time
            ) / 60

            print(
                f"Processed {gene_idx + 1:,} genes | "
                f"NNZ={indptr[-1]:,} | "
                f"{elapsed:.1f} min"
            )

print("\nStreaming complete.")
print("Genes:", len(gene_names))
print("Cells:", n_cells_expected)
print("Non-zero values:", f"{indptr[-1]:,}")

In [ ]:
print("=" * 80)
print("STEP18 CORRECTED CHECKPOINT SUMMARY")
print("=" * 80)

# 1. Metadata-only cells
print("\n[1] Metadata-only cells")
print(metadata_only_cells.tolist())

# 2. Count-only cells
print("\n[2] Count-only cells")
print(count_only_cells.tolist())

# 3. Match rate
print("\n[3] Cell-metadata match")
print("Count matrix cells:", len(count_cells))
print("Matched metadata cells:", len(common_cells))
print(f"Match rate: {match_rate * 100:.4f}%")

# 4. Streaming result
print("\n[4] Streaming count matrix result")
print("Genes:", len(gene_names))
print("Cells:", n_cells_expected)
print("Non-zero values:", f"{indptr[-1]:,}")

print("\n" + "=" * 80)
print("END OF SUMMARY")
print("=" * 80)

In [ ]:
# ============================================================
# STEP18 - Build Gene × Cell sparse matrix
# ============================================================

from scipy import sparse
import numpy as np

if len(data_parts) > 0:
    data = np.concatenate(data_parts)
    indices = np.concatenate(index_parts)
else:
    data = np.array([], dtype=np.int32)
    indices = np.array([], dtype=np.int32)

indptr_arr = np.asarray(
    indptr,
    dtype=np.int64
)

X_gene_cell = sparse.csr_matrix(
    (
        data,
        indices,
        indptr_arr
    ),
    shape=(
        len(gene_names),
        n_cells_expected
    ),
    dtype=np.int32
)

print("=" * 80)
print("GENE × CELL SPARSE MATRIX")
print("=" * 80)

print("Shape:", X_gene_cell.shape)
print("NNZ:", f"{X_gene_cell.nnz:,}")

density = (
    X_gene_cell.nnz /
    (
        X_gene_cell.shape[0]
        * X_gene_cell.shape[1]
    )
)

print(f"Density: {density * 100:.4f}%")

In [ ]:
# ============================================================
# STEP18 - Release temporary construction memory
# ============================================================

import gc

del data_parts
del index_parts
del data
del indices
del indptr
del indptr_arr

gc.collect()

print("Temporary construction memory released.")

In [ ]:
# ============================================================
# STEP18 - Convert to Cell × Gene
# ============================================================

X_cell_gene = (
    X_gene_cell
    .T
    .tocsr()
)

print("=" * 80)
print("CELL × GENE MATRIX")
print("=" * 80)

print("Shape:", X_cell_gene.shape)
print("NNZ:", f"{X_cell_gene.nnz:,}")

In [ ]:
# ============================================================
# STEP18 - Construct AnnData
# ============================================================

import anndata as ad
import pandas as pd

adata = ad.AnnData(
    X=X_cell_gene,

    obs=meta2.loc[
        count_cells
    ].copy(),

    var=pd.DataFrame(
        index=pd.Index(
            gene_names,
            dtype="object"
        )
    )
)

adata.obs_names = (
    count_cells.astype(str)
)

print(adata)

In [ ]:
# ============================================================
# STEP18 - AnnData alignment audit
# ============================================================

print("=" * 80)
print("ANNDATA ALIGNMENT AUDIT")
print("=" * 80)

print("Cells:", adata.n_obs)
print("Genes:", adata.n_vars)

print("\nFirst 5 cells:")
print(adata.obs_names[:5].tolist())

print("\nFirst 5 genes:")
print(adata.var_names[:5].tolist())

print("\nFirst 5 metadata rows:")
display(
    adata.obs[
        [
            "sample",
            "site",
            "patient",
            "stage",
            "virus",
            "celltype"
        ]
    ].head()
)

assert adata.n_obs == 71915
assert adata.n_vars == 25712
assert adata.obs_names.is_unique

print("\nANNDATA ALIGNMENT = PASS")

In [ ]:
# ============================================================
# STEP18 - Gene-name duplication audit
# ============================================================

duplicate_gene_n = int(
    adata.var_names
    .duplicated()
    .sum()
)

print("Duplicated gene names:", duplicate_gene_n)

if duplicate_gene_n > 0:

    duplicated_genes = (
        adata.var_names[
            adata.var_names.duplicated(
                keep=False
            )
        ]
        .tolist()
    )

    pd.DataFrame({
        "Gene": duplicated_genes
    }).to_csv(
        RESULTS_DIR /
        "Duplicated_Gene_Names.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "Duplicated gene list saved."
    )

adata.var_names_make_unique()

print(
    "Unique var names after processing:",
    adata.var_names.is_unique
)

In [ ]:
# ============================================================
# STEP18 - Bulk anchor coverage
# ============================================================

ORIGINAL_TOP10 = [
    "HOXD9",
    "RFESD",
    "RPSAP53",
    "TRGC1",
    "FAM227B",
    "AC015871.6",
    "LINC00324",
    "ACOXL",
    "PTMAP4",
    "ANKRD20A7P",
]

HUB18 = [
    "CENPA",
    "KIF2C",
    "CCNB1",
    "CDC20",
    "PRR11",
    "CDCA8",

    "SLC52A2",
    "RPL8",
    "BOP1",
    "ZFAS1",
    "SNHG6",
    "RPL7P9",

    "CYP8B1",
    "GLYATL1",
    "PLG",
    "ACSM2A",
    "ABAT",
    "MAT1A",
]


def gene_coverage(adata, gene_list, name):

    available = set(adata.var_names)

    present = [
        g for g in gene_list
        if g in available
    ]

    missing = [
        g for g in gene_list
        if g not in available
    ]

    print("=" * 70)
    print(name)
    print(
        f"Coverage: "
        f"{len(present)}/{len(gene_list)}"
    )

    print("\nPresent:")
    print(present)

    print("\nMissing:")
    print(missing)

    return {
        "FeatureSet": name,
        "Total": len(gene_list),
        "Present": len(present),
        "Missing": len(missing),
        "PresentGenes":
            ";".join(present),
        "MissingGenes":
            ";".join(missing)
    }


coverage_df = pd.DataFrame([
    gene_coverage(
        adata,
        ORIGINAL_TOP10,
        "OriginalTop10"
    ),

    gene_coverage(
        adata,
        HUB18,
        "WGCNA_Hub18"
    )
])

coverage_df.to_csv(
    RESULTS_DIR /
    "Bulk_Anchor_Gene_Coverage.csv",
    index=False,
    encoding="utf-8-sig"
)

display(coverage_df)

In [ ]:
# ============================================================
# STEP18 - Save raw unfiltered AnnData
# ============================================================

raw_h5ad = (
    RESULTS_DIR /
    "GSE149614_RAW_UNFILTERED.h5ad"
)

adata.write_h5ad(
    raw_h5ad,
    compression="gzip"
)

print("Saved:")
print(raw_h5ad)

print(
    "File size GB:",
    round(
        raw_h5ad.stat().st_size /
        1024**3,
        3
    )
)

In [ ]:
# ============================================================
# STEP18 - Final audit report
# ============================================================

import json

audit_report = {

    "Dataset":
        "GSE149614",

    "Role":
        "Primary scRNA discovery cohort",

    "Stage":
        "Step18_Data_Audit",

    "Status":
        "COMPLETE",

    "Cells":
        int(adata.n_obs),

    "Genes":
        int(adata.n_vars),

    "NNZ":
        int(adata.X.nnz),

    "Sparse_density_pct":
        float(
            adata.X.nnz /
            (
                adata.n_obs *
                adata.n_vars
            ) * 100
        ),

    "Barcode_column":
        "Cell",

    "Barcode_match_rate":
        1.0,

    "OriginalTop10_coverage":
        int(
            sum(
                g in adata.var_names
                for g in ORIGINAL_TOP10
            )
        ),

    "Hub18_coverage":
        int(
            sum(
                g in adata.var_names
                for g in HUB18
            )
        ),

    "Raw_counts_preserved":
        True,

    "QC_performed":
        False,

    "Normalization_performed":
        False,

    "Integration_performed":
        False,

    "Output_h5ad":
        str(raw_h5ad),

    "Next_step":
        "Step19_scRNA_QC"
}


with open(
    RESULTS_DIR /
    "Step18_Data_Audit.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        audit_report,
        f,
        ensure_ascii=False,
        indent=4
    )

print(
    json.dumps(
        audit_report,
        ensure_ascii=False,
        indent=4
    )
)

In [ ]:
# ============================================================
# STEP18 - Final checkpoint
# ============================================================

print("=" * 80)

print("STEP18 STATUS = COMPLETE")

print("\nPRIMARY scRNA DATASET:")
print("GSE149614")

print("\nCELLS:")
print(f"{adata.n_obs:,}")

print("\nGENES:")
print(f"{adata.n_vars:,}")

print("\nORIGINAL TOP10 COVERAGE:")
print(
    f"{sum(g in adata.var_names for g in ORIGINAL_TOP10)}/10"
)

print("\nHUB18 COVERAGE:")
print(
    f"{sum(g in adata.var_names for g in HUB18)}/18"
)

print("\nRAW COUNTS = PRESERVED")
print("QC = NOT PERFORMED")
print("NORMALIZATION = NOT PERFORMED")

print("\nNEXT STEP:")
print("Step19 — scRNA Quality Control")

print("=" * 80)

In [ ]:
# ============================================================
# STEP18 - Record single-cell anchor interpretation
# ============================================================

anchor_interpretation = pd.DataFrame([
    {
        "Feature_Set": "OriginalTop10",
        "Total": 10,
        "Detected": 6,
        "Missing": 4,
        "Role_in_scRNA":
            "Biological anchor only; do NOT reconstruct the full frozen Cox RiskScore",
        "Scoring_policy":
            "Use detectable members only, clearly labeled as detectable-gene score"
    },
    {
        "Feature_Set": "WGCNA_Hub18",
        "Total": 18,
        "Detected": 16,
        "Missing": 2,
        "Role_in_scRNA":
            "Primary cross-scale biological anchor for malignant-state characterization",
        "Scoring_policy":
            "Use detectable members with module-specific scoring"
    }
])

anchor_interpretation.to_csv(
    RESULTS_DIR / "SingleCell_Anchor_Interpretation.csv",
    index=False,
    encoding="utf-8-sig"
)

display(anchor_interpretation)

In [ ]:
print("=" * 80)
print("STEP18 FINAL BASIC CHECK")
print("=" * 80)

# 1. AnnData shape
print("\n[1] AnnData shape")
print(adata.shape)
print(f"Cells: {adata.n_obs:,}")
print(f"Genes: {adata.n_vars:,}")

# 2. Duplicated gene names
print("\n[2] Duplicated gene names")

duplicate_gene_n = int(
    adata.var_names.duplicated().sum()
)

print("Duplicated gene names:", duplicate_gene_n)

if duplicate_gene_n > 0:
    duplicated_genes = (
        adata.var_names[
            adata.var_names.duplicated(keep=False)
        ]
        .tolist()
    )

    print("\nFirst 20 duplicated gene names:")
    print(duplicated_genes[:20])

print("\n" + "=" * 80)
print("END")
print("=" * 80)